In [1]:
# Must come before TF import
import os
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"

In [2]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, f1_score,
)

sys.path.insert(0, str(Path("pipelines").resolve()))

from pipeline_utils import (
    load_and_prepare_data, build_all_splits,
    N_CLASSES, WINDOW_SIZE,
)
from rf_pipeline import features_fft, build_feature_matrix

print("Working dir:", Path.cwd())

/Users/kacharino/myProjects/Bachelor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working dir: /Users/kacharino/myProjects/Bachelor


In [3]:
# Resample 100→50 Hz, majority window labels, z-score norm (train-only fit)
session_data, mean_vec, std_vec = load_and_prepare_data()
splits = build_all_splits(session_data)

unseen_sess, unseen_start, unseen_lab = splits["unseen"]

MAX_WIN = 10_000
n = min(len(unseen_lab), MAX_WIN)
print(f"Using {n:,} of {len(unseen_lab):,} unseen windows.")

# Materialise windows as float32 numpy array — required before predict()
X_unseen = np.empty((n, WINDOW_SIZE, 6), dtype=np.float32)
for i in range(n):
    sid = int(unseen_sess[i])
    st  = int(unseen_start[i])
    X_unseen[i] = session_data[sid]["x"][st : st + WINDOW_SIZE]

y_true = unseen_lab[:n].astype(np.int32)
print(f"X_unseen: {X_unseen.shape}, dtype={X_unseen.dtype}")

Datenpipeline startet...
  Session w00 geladen: (106765, 6)
  Session w01 geladen: (123109, 6)
  Session w02 geladen: (123388, 6)
  Session w03 geladen: (105817, 6)
  Session w04 geladen: (97985, 6)
  Session w05 geladen: (95798, 6)
  Session w06 geladen: (91508, 6)
  Session w07 geladen: (118237, 6)
  Session w08 geladen: (96533, 6)
  Session w09 geladen: (176354, 6)
  Session w10 geladen: (130439, 6)
  Session w11 geladen: (104356, 6)
  Session w12 geladen: (113066, 6)
  Session w13 geladen: (131671, 6)
  Session w14 geladen: (80870, 6)
  Session w15 geladen: (95182, 6)
  Session w16 geladen: (128992, 6)
  Session w17 geladen: (108032, 6)
  Session w18 geladen: (200085, 6)
  Session w19 geladen: (86054, 6)
  Session w20 geladen: (155209, 6)
Normalisierung abgeschlossen (train-only fit).
Splits: Train=119123 | Val=26138 | Seen=41041 | Unseen=60128
Using 10,000 of 60,128 unseen windows.
X_unseen: (10000, 250, 6), dtype=float32


In [4]:
CLASSES = [
    'squats', 'lunges', 'bicep_curls', 'sit_ups', 'push_ups',
    'tricep_extensions', 'dumbbell_rows', 'jumping_jacks',
    'dumbbell_shoulder_press', 'lateral_raises', 'non_exercise',
]

def plot_and_save_cm(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred,
                          labels=list(range(N_CLASSES)),
                          normalize="true")
    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    disp.plot(ax=ax, cmap="Blues", colorbar=True, values_format=".2f")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    plt.xticks(rotation=40, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

## CNN V3-C — Primary path (Keras, compile=False)
Run this cell. If it takes > 3 minutes, interrupt and run the **CoreML fallback** cell below instead.

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "pipelines/best_cnn_v3c.keras",
    compile=False,
)
print("Model loaded (compile=False).")

probs      = model.predict(X_unseen, batch_size=1024, verbose=1)
y_pred_cnn = np.argmax(probs, axis=1).astype(np.int32)

print(f"\nCNN macro-F1: {f1_score(y_true, y_pred_cnn, average='macro'):.4f}  (expected ~0.8946)")
print(classification_report(y_true, y_pred_cnn, target_names=CLASSES, digits=4))

plot_and_save_cm(
    y_true, y_pred_cnn,
    title="CNN V3-C \u2014 Confusion Matrix (Unseen Test Set)",
    save_path="confusion_matrix_cnn_v3c.png",
)

/Users/kacharino/myProjects/Bachelor/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Model loaded (compile=False).


## CNN V3-C — Fallback: CoreML inference on Apple Neural Engine
Only run this if the Keras cell above timed out. CoreML runs on the ANE and is typically 5-10× faster on Apple Silicon.

In [1]:
import coremltools as ct

cnn_coreml = ct.models.MLModel(
    "pipelines/CNNModel_v3c.mlpackage",
    compute_units=ct.ComputeUnit.CPU_AND_NE,  # CPU + Neural Engine, no GPU needed
)
print("CoreML model loaded.")

# Model expects input name 'x', shape (1, 250, 6)
# Run in batches of 256 for progress visibility
BATCH = 256
y_pred_coreml = np.empty(n, dtype=np.int32)

for start in range(0, n, BATCH):
    end   = min(start + BATCH, n)
    chunk = X_unseen[start:end]          # (B, 250, 6)
    preds = np.empty(end - start, dtype=np.int32)
    for j, sample in enumerate(chunk):
        inp    = sample[np.newaxis, ...]  # (1, 250, 6)
        out    = cnn_coreml.predict({"x": inp})
        preds[j] = int(np.argmax(out["clf_softmax"]))
    y_pred_coreml[start:end] = preds
    if (start // BATCH) % 4 == 0:
        print(f"  {end}/{n} windows predicted...")

print(f"\nCNN (CoreML) macro-F1: {f1_score(y_true, y_pred_coreml, average='macro'):.4f}  (expected ~0.8946)")
print(classification_report(y_true, y_pred_coreml, target_names=CLASSES, digits=4))

plot_and_save_cm(
    y_true, y_pred_coreml,
    title="CNN V3-C \u2014 Confusion Matrix (Unseen Test Set)",
    save_path="confusion_matrix_cnn_v3c.png",
)

scikit-learn version 1.8.0 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
/Users/kacharino/myProjects/Bachelor/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
TensorFlow version 2.20.0 has not been tested with coremltools. You may run into unexpected errors. TensorFlow 2.12.0 is the most recent version that has been tested.


CoreML model loaded.


NameError: name 'np' is not defined

## RF Optuna — Unseen Test Set

In [ ]:
import joblib

rf_model = joblib.load("pipelines/best_rf_optuna.joblib")
print("RF Optuna loaded.")

print("Computing FFT features (114 per window)...")
X_rf = build_feature_matrix(
    session_data,
    unseen_sess[:n],
    unseen_start[:n],
    features_fft,
)
print(f"Feature matrix: {X_rf.shape}")

y_pred_rf = rf_model.predict(X_rf).astype(np.int32)

print(f"\nRF macro-F1: {f1_score(y_true, y_pred_rf, average='macro'):.4f}  (expected ~0.7831)")
print(classification_report(y_true, y_pred_rf, target_names=CLASSES, digits=4))

plot_and_save_cm(
    y_true, y_pred_rf,
    title="RF Optuna \u2014 Confusion Matrix (Unseen Test Set)",
    save_path="confusion_matrix_rf_optuna.png",
)